In [0]:
catalog = '`de-foundation-01`'

In [0]:
from pyspark.sql.functions import (
    sum,
    last_day,
    col
)

actual_monthly = (
    spark.table(
        f"{catalog}.silver.actual_cost"
    )
    .withColumn(
        "Month_End_Date",
        last_day(col("Cost_Date_Clean"))
    )
    .groupBy(
        "Project_Code",
        "Month_End_Date"
    )
    .agg(
        sum("Amount_Clean")
        .alias("Actual_Cost")
    )
)

In [0]:
budget_monthly = (
    spark.table(
        f"{catalog}.silver.budget"
    )
    .groupBy(
        "Project_Code",
        "Month_End_Date"
    )
    .agg(
        sum("Budget_Amount")
        .alias("Budget")
    )
)


In [0]:
project = (
    spark.table(f"{catalog}.silver.project")
    .select(
        "Project_Code",
        "Project_Name"
    )
)

gold = (
    actual_monthly
    .join(
        budget_monthly,
        ["Project_Code", "Month_End_Date"],
        "full"
    )
    .join(
        project,
        "Project_Code",
        "left"
    )
    .fillna(
        0,
        subset=[
            "Actual_Cost",
            "Budget"
        ]
    )
    .withColumn(
        "Variance",
        col("Budget") - col("Actual_Cost")
    )
)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{catalog}.gold.project_monthly_performance"
    )
)